<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.4: FIRRTL 转换示例

**上一节: [常见转换模式](4.3_firrtl_common_idioms.ipynb)**<br>

这个 AnalyzeCircuit 转换会遍历 `firrtl.ir.电路`，并记录每个模块中找到的加法操作数量。

## 设置

请运行以下代码:

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
// 编译器基础架构

// Firrtl IR 类

// Map 函数

// Scala 的可变集合
import scala.collection.mutable



## 统计每个模块的加法器数量

如前所述，Firrtl 电路使用树形表示：
  - 一个 Firrtl `电路` 包含一系列 `DefModule`。
  - 一个 `DefModule` 包含一系列 `Port`，可能还有一个 `Statement`。
  - 一个 `Statement` 可以包含其他 `Statement` 或 `Expression`。
  - 一个 `Expression` 可以包含其他 `Expression`。

要访问电路中的所有 Firrtl IR 节点，我们编写递归遍历这棵树的函数。为了记录统计信息，我们将传递一个 `Ledger` 类，并在遇到加法操作时使用它：

In [ ]:
class Ledger {
  import firrtl.Utils
  private var moduleName: Option[String] = None
  private val modules = mutable.Set[String]()
  private val moduleAddMap = mutable.Map[String, Int]()
  def foundAdd(): Unit = moduleName match {
    case None => sys.error("Module name not defined in Ledger!")
    case Some(name) => moduleAddMap(name) = moduleAddMap.getOrElse(name, 0) + 1
  }
  def getModuleName: String = moduleName match {
    case None => Utils.error("Module name not defined in Ledger!")
    case Some(name) => name
  }
  def setModuleName(myName: String): Unit = {
    modules += myName
    moduleName = Some(myName)
  }
  def serialize: String = {
    modules map { myName =>
      s"$myName => ${moduleAddMap.getOrElse(myName, 0)} add ops!"
    } mkString "\n"
  }
}

现在，让我们定义一个 FIRRTL 转换，它会遍历电路并在遇到加法器（带有 `Add` 操作参数的 `DoPrim`）时更新我们的 `Ledger`。暂时不用担心 `inputForm` 或 `outputForm`。

花些时间理解 `walkModule`、`walkStatement` 和 `walkExpression` 如何实现对 FIRRTL AST 中所有 `DefModule`、`Statement` 和 `Expression` 节点的遍历。

需要回答的问题：
  - **为什么 walkModule 不调用 walkExpression？**
  - **为什么 walkExpression 进行后序遍历？**
  - **你能修改 walkExpression 以进行表达式的前序遍历吗？**

In [ ]:
class AnalyzeCircuit extends firrtl.Transform {
  import firrtl._
  import firrtl.ir._
  import firrtl.Mappers._
  import firrtl.Parser._
  import firrtl.annotations._
  import firrtl.PrimOps._
    
  // 要求 [[Circuit]] 形式为 "low"
  def inputForm = LowForm
  // 指示输出 [[Circuit]] 形式为 "low"
  def outputForm = LowForm

  // 由 [[Compiler]] 调用以运行你的传递。[[CircuitState]] 包含
  // 电路及其形式，以及其他相关数据。
  def execute(state: CircuitState): CircuitState = {
    val ledger = new Ledger()
    val circuit = state.circuit

    // 对电路中的每个 [[DefModule]] 执行 walkModule(ledger) 函数，
    // 返回一个带有新 [[DefModule]] 序列的新 [[Circuit]]。
    //   - "高阶函数" - 将函数用作对象
    //   - "函数柯里化" - 部分参数表示法
    //   - "中缀表示法" - 花哨的函数调用语法
    //   - "map" - 经典函数式编程概念
    //   - 丢弃返回的新 [[Circuit]]，因为电路未被修改
    circuit map walkModule(ledger)

    // 打印我们的 ledger
    println(ledger.serialize)

    // 返回未更改的 [[CircuitState]]
    state
  }

  // 深度访问 m 中的每个 [[Statement]]。
  def walkModule(ledger: Ledger)(m: DefModule): DefModule = {
    // 将 ledger 设置为当前模块名称
    ledger.setModuleName(m.name)

    // 对 m 中的每个 [[Statement]] 执行 walkStatement(ledger) 函数。
    //   - 返回新的 [[DefModule]]（此处与 m 相同）
    //   - 如果 m 不包含 [[Statement]]，map 返回 m。
    m map walkStatement(ledger)
  }

  // 深度访问 s 中的每个 [[Statement]] 和 [[Expression]]。
  def walkStatement(ledger: Ledger)(s: Statement): Statement = {

    // 对 s 中的每个 [[Expression]] 执行 walkExpression(ledger) 函数。
    //   - 丢弃新的 [[Statement]]（此处与 s 相同）
    //   - 如果 s 不包含 [[Expression]]，map 返回 s。
    s map walkExpression(ledger)

    // 对 s 中的每个 [[Statement]] 执行 walkStatement(ledger) 函数。
    //   - 返回新的 [[Statement]]（此处与 s 相同）
    //   - 如果 s 不包含 [[Statement]]，map 返回 s。
    s map walkStatement(ledger)
  }

  // 深度访问 e 中的每个 [[Expression]]。
  //   - "后序遍历" - 在处理 e 之前先处理其子节点 [[Expression]]
  def walkExpression(ledger: Ledger)(e: Expression): Expression = {

    // 对 e 中的每个 [[Expression]] 执行 walkExpression(ledger) 函数。
    //   - 返回新的 [[Expression]]（此处与 e 相同）
    //   - 如果 s 不包含 [[Expression]]，map 返回 e。
    val visited = e map walkExpression(ledger)

    visited match {
      // 如果 e 是加法器，则增加我们的 ledger 并返回 e。
      case DoPrim(Add, _, _, _) =>
        ledger.foundAdd
        e
      // 如果 e 不是加法器，则返回 e。
      case notadd => notadd
    }
  }
}

## 运行我们的转换

既然我们已经定义了它，让我们在一个 Chisel 设计上运行它！首先，让我们定义一个 Chisel 模块。

In [ ]:
// Chisel 相关内容
import chisel3._
import chisel3.Input // 技术细节：避免与 _root_.almond.input.Input 冲突
import chisel3.util._

In [ ]:
class AddMe(val nInputs: Int, val width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Vec(nInputs, UInt(width.W)))
    val out = Output(UInt(width.W))
  })
  io.out := io.in.reduce(_ +& _)
}

接下来，让我们将其详细化为 FIRRTL AST 语法。

In [ ]:
val firrtlSerialization = chisel3.Driver.emit(() => new AddMe(8, 4))

最后，让我们将 FIRRTL 编译为 Verilog，但在编译中包含我们的自定义转换。请注意它会打印出找到的加法操作数量！

**注意** (2021年1月): 以下行可能由于[bug](https://github.com/freechipsproject/Chisel-bootcamp/issues/129)而无法正常工作。

In [ ]:
val verilog = compileFIRRTL(firrtlSerialization, new firrtl.VerilogCompiler(), Seq(new AnalyzeCircuit()))

`compileFIRRTL` 函数仅在此教程中定义 - 在后续章节中，我们将描述如何插入自定义转换的过程。

本节到此结束！